# The Grid Module
The grid module provides an interface for library and user-defined modules to add grid points to the NLP.  Despite its name, the grid module does not create its own grid points.  Its primary role is to partition the grid into phases and subphases.  It also provides a mechanism for built-in and custom modules to add their grid points to the NLP.  The model definition of the grid module is presented at the end of this notebook.

**NOTE:** In Amplify, nearly every module depends on the grid module.

In [ ]:
from amplpy import AMPL
ampl = AMPL()

The grid module is split into two blocks of code.  The main block in the cell below defines the general behavior of how modules add grid points to the NLP.  A detailed breakdown of the code along with working examples can be found in the other grid-related notebooks.

In [ ]:
%%ampl_eval
#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#* grid
#----------- modules
# model-defined modules;
set MOD_MODULES;
set MOD_GRID {MOD_MODULES}; # this set will be redeclared later

# data-defined modules
set DAT_MODULES default {};
set DAT_GRID {DAT_MODULES};

# all modules
set MODULES = MOD_MODULES union DAT_MODULES;
set GRID_MODULES {i1 in MODULES} := 
  if i1 in MOD_MODULES then MOD_GRID[i1]  
  else DAT_GRID[i1];

#----------- phases
# dynamics are constant within a phase
set SUBPHASES within Reals;
set SUBPHASES_MBRS = {'PHASE', 'BEG', 'END'};
param subphases {SUBPHASES, SUBPHASES_MBRS};

# subphases must map to phase through floor function
check {i1 in SUBPHASES} : subphases[i1, 'PHASE'] = floor(i1);
# beginning must be less than end
check {i1 in SUBPHASES} : subphases[i1, 'BEG'] <= subphases[i1, 'END'];
# make sure sets don't overlap
check {i1 in SUBPHASES, i2 in SUBPHASES : i1 != i2} : inter {i3 in {i1, i2}} union {i4 in {'BEG', 'END'}} {subphases[i3, i4]} within {};

set PHASES = setof {i1 in SUBPHASES} subphases[i1, 'PHASE'];

#----------- grid points
param GRID_MAX >= 0;

set GRID_MBRS = {0} union SUBPHASES union PHASES;

set GRID {i1 in GRID_MBRS} ordered by [0, GRID_MAX] :=
  if i1 = 0 then # add grid points from registered modules
    union {i2 in MODULES} GRID_MODULES[i2]
  else if i1 in Integers then # collect grid points for each phase
    union {i2 in SUBPHASES : floor(i2) = i1} GRID[i2]
  else {i2 in GRID[0] : subphases[i1, 'BEG'] <= i2 <= subphases[i1, 'END']};  # collect grid points for each subphase

#----------- decision variable: time
var t {GRID[0]}; # time
#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#* grid!

Below is a hack to allow Amplify to get around AMPL's declaration rules.  The main block above defines how grid points are added to the NLP.  The block below specifies how modules like the Runge-Kutta and Bezier modules add their specific grid points using the sets and parameters defined within their respective modules.

**IMPORTANT:** For maximum portability, my preference is that only Amplify extends the code below to include other modules of its choosing.  All others should add their module grid points in the data file.

In [ ]:
#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#* grid
# model has been processed, so now add data specific rules for modules in the model file
redeclare set MOD_GRID {i1 in MOD_MODULES} := 
  if i1 = 'ODE' then ODE_GRID
  else if i1 = 'PIM' then union {(i2,i3) in PIM} {i2, i3}
  else if i1 = 'BEZ' then union {i2 in BEZ_LBLS, i3 in BEZ_SUBGRID[i2]} bez_subgrid[i2, i3, 'BEG']..bez_subgrid[i2, i3, 'END']
  else {};
#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#* grid!